# Research questions — buying accuracy with FLOPs

## What this is

You have built an autograd engine, layers, two trained models (an Adult income
classifier in [q05](../exercises/q05_binary_classification.ipynb), word2vec embeddings in
[q06](../exercises/q06_learn_embedding.ipynb)), and you can read every number that comes
out of them. This notebook is the step after that: **pick one of the fifteen questions
below and spend two to five days trying to answer it experimentally.**

These are not exercises with a key at the back. Each one is a question whose answer *in
this setting* — pure NumPy, CPU, a 108-feature tabular task and a small book — is not
written down anywhere. The literature tells you what happened at the scale of ImageNet or
of a 7-billion-parameter language model; whether the same thing happens on 26 049 rows and
a 109×64 matrix is genuinely open, and it is small enough that you can find out.

## The one axis they share

Every question is posed as a **trade-off against compute**, because "does this idea help?"
is unanswerable in a week while "does this idea buy accuracy per FLOP?" is answerable in
two days. You can ask it here because the engine counts its own arithmetic:

```python
cpu.reset_flops()
...                      # anything built from engine ops
cost = cpu.flop_count()  # exact for the matmuls that dominate
```

That counter is the instrument of this whole project. The baselines it reports — the
numbers every answer is measured against — are:

| baseline | score | cost |
|---|---|---|
| q05 Adult MLP (108→64→ReLU→2, Adam, 100 epochs) | test accuracy **0.8543** | **84.23 GFLOP** to train, 232 MFLOP per inference pass |
| always answer `<=50K` | test accuracy **0.7592** | 0 |
| q06 word2vec (D=32, K=5, 100 epochs) | silhouette **−0.058 → +0.141** | **6.6 GFLOP** |

## How you work

1. **Choose one question.** Read its papers first — a day of reading is part of the five.
2. **Reproduce the baseline** with `research/benchmark.py` before changing anything. If you
   cannot reproduce 0.8543, stop: something in your setup is off and every later number
   would be noise.
3. **Change one thing.** Implement the variation, sweep the one or two knobs the question
   is about, measure accuracy *and* FLOPs, over at least three seeds.
4. **Write the report** (template at the end). A negative result, measured properly, is a
   complete answer — and is worth more than a positive result from one seed.

## The measurement protocol

The rules below are what separate a result from an anecdote. They are also the bulk of the
grade.

**Three seeds, minimum.** Run every configuration with at least `seeds=(0, 1, 2)` and
report **mean ± standard deviation**. The effects these questions chase are often smaller
than the spread between seeds — discovering that is a finding, not a failure.

**Never compare two numbers whose error bars overlap.** If your variant scores
0.8551 ± 0.0012 and the baseline scores 0.8543 ± 0.0009, you have *not* shown an
improvement. Say so plainly, then either run more seeds or accept the null result.

**Always report the cost beside the score.** An accuracy without a FLOP count answers none
of these questions. Two costs matter and they are not interchangeable:

- **training FLOPs** — everything the engine executed while fitting. Quote this when the
  question is about the cost of *learning* (optimisers, schedules, data pruning, SSL).
- **inference FLOPs** — one forward pass over the test split. Quote this when the question
  is about the cost of *using* the model (low-rank, quantisation, sparsity, distillation,
  conditional compute).

**Hold the right thing fixed.** Comparing per-step ideas (a loss, a regulariser)? Fix the
number of epochs. Comparing training schemes (an optimiser, a schedule, a data diet)? Fix
the **total training FLOP budget** and let the epochs fall where they may. Saying which one
you fixed is part of the claim.

**Be honest about what the counter does not count.** `engine.py` charges the forward cost
of every op and the *matmul* backward — the dominant term — so the tally is exact for the
matrix products and approximate for the cheap elementwise ops, and it counts a masked or
ternary matmul at full price. If your question turns on that (Q2, Q3, Q6), you must say how
you accounted for it: either extend the counter, or report an *effective* FLOP count you
compute yourself and justify.

**Keep the comparison in one table.** Your variant and the baseline, same table, same
seeds, same code path. `summarize()` does this for you and flags whether a difference is
inside one standard deviation.

In [ ]:
# Run me first: make the project importable whether Jupyter started in the project
# root or inside research/.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "bert_cpu").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

from bert_cpu import engine as cpu
from bert_cpu import nn
from bert_cpu import optim
from bert_cpu.loss import cross_entropy
from research.benchmark import (MLP, baseline_mlp, count_flops, pareto, repeat,
                                run_adult, run_sgns, summarize)

print("ready — project root:", ROOT)

### The instrument, in one cell

`research/benchmark.py` is the measuring device: it runs the q05 loop (or the q06 loop) and
hands back the score *with* its cost, for one seed. `repeat` gives you the seeds,
`summarize` the table, `pareto` the picture. Read that file before you start — it is
short, and knowing exactly what it counts is half of measuring well.

The cell below is the shape **every** answer in this notebook takes: define a variation,
run it beside the baseline, look at the table. It takes about 30 seconds.

In [ ]:
class LowRankMLP(nn.Module):
    """Example variation: the first layer factored through a rank-r bottleneck."""

    def __init__(self, n_features: int, rank: int = 8, hidden: int = 64) -> None:
        self.u = nn.Linear(n_features, rank)
        self.v = nn.Linear(rank, hidden)
        self.fc2 = nn.Linear(hidden, 2)

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        return self.fc2(self.v(self.u(x)).relu())


rows = []
rows += repeat(lambda s: run_adult(baseline_mlp, seed=s, name="baseline"))
rows += repeat(lambda s: run_adult(lambda n: LowRankMLP(n, rank=8), seed=s, name="low-rank r=8"))

print(summarize(rows, baseline="baseline"))

In [ ]:
# The same rows as a picture: score against what it cost.
pareto(rows, cost="infer_flops", title="Adult: accuracy vs inference cost")

---

## The fifteen questions

Pick **one**. The difficulty stars are about implementation effort, not about how
interesting the answer is — ★ questions are just as publishable as ★★★ ones if measured
well.

| # | question | data | what you implement | ★ |
|---|---|---|---|---|
| 1 | [Low-rank layers](#q1) | Adult | a factored `Linear` | ★ |
| 2 | [Ternary weights](#q2) | Adult | a quantise op with a straight-through backward | ★★ |
| 3 | [Sparse training: prune or grow?](#q3) | Adult | masks + a prune/grow schedule | ★★★ |
| 4 | [Conditional compute](#q4) | Adult | a top-1 router over small experts | ★★★ |
| 5 | [Star vs. sum at equal FLOPs](#q5) | Adult | q02's `StarLinear`, widened | ★ |
| 6 | [Activation sparsity](#q6) | Adult | ReLU², a sparsity meter, an effective-FLOP count | ★★ |
| 7 | [Do learned activation mixtures pay?](#q7) | Adult | q04's `LearnableActivation`, trained | ★ |
| 8 | [Loss shape under imbalance](#q8) | Adult | focal / label-smoothing / poly losses | ★★ |
| 9 | [R-Drop vs. training longer](#q9) | Adult | the `nn.Dropout` stub + a KL term | ★★ |
| 10 | [Mixup on one-hot rows](#q10) | Adult | input- and hidden-space mixup | ★★ |
| 11 | [Optimisers at equal budget](#q11) | Adult | Lion and/or Muon, from the papers | ★★ |
| 12 | [Distil a wide teacher](#q12) | Adult | a KD loss on soft targets | ★★ |
| 13 | [How much data can you throw away?](#q13) | Adult | EL2N / forgetting scores | ★★ |
| 14 | [Pretrain without labels](#q14) | Adult | SCARF corruption + InfoNCE + a linear probe | ★★★ |
| 15 | [The shape of an embedding table](#q15) | Flatland | sweeps over D, K, tying, subsampling | ★★ |

<a id="q1"></a>
## 1. Low-rank layers — is a wide bottleneck better than a narrow layer?

**The idea.** In the q05 model the first weight matrix is $109\times64$ and it costs
$2 \cdot (64 \cdot 26\,049) \cdot 109 \approx 364$ MFLOP per forward pass — about **98 %**
of the whole model's arithmetic. Factor it, $\mathbf{W} \approx \mathbf{U}\mathbf{V}$ with
$\mathbf{U} \in \mathbb{R}^{109\times r}$ and $\mathbf{V} \in \mathbb{R}^{r\times 64}$, and
the cost becomes proportional to $r(109 + 64)$ instead of $109 \cdot 64$. The saving is
real; the question is what it costs in accuracy, and whether the freed compute is better
spent on width.

**Readings**
- Denil, Shakibi, Dinh, Ranzato & de Freitas, *Predicting Parameters in Deep Learning*,
  NeurIPS 2013 — [arXiv:1306.0543](https://arxiv.org/abs/1306.0543). The original claim
  that most weights are redundant.
- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022 —
  [arXiv:2106.09685](https://arxiv.org/abs/2106.09685).
- Lan et al., *ALBERT*, ICLR 2020 — [arXiv:1909.11942](https://arxiv.org/abs/1909.11942).
  §3: factorised embedding parameterisation, the same trick on a vocabulary table.
- Dao et al., *Monarch: Expressive Structured Matrices for Efficient and Accurate
  Training*, ICML 2022 — [arXiv:2204.00595](https://arxiv.org/abs/2204.00595). What to read
  if you want a structured alternative to plain low rank.

**The open question.** *At a fixed inference-FLOP budget, is a wide low-rank layer better
than a narrow full-rank one?* And: does a network **trained** factored match one trained
dense and factored **afterwards** by SVD?

**What to implement.** `LowRankMLP` is already in the example cell above — that part is
free. The work is the sweep, plus the post-hoc arm: train the baseline, take the SVD of
`model.fc1.weight.data`, truncate to rank $r$, and measure the damage before and after a
short fine-tune.

**Protocol.** Sweep $r \in \{2,4,8,16,32,64\}$ against `hidden` $\in \{32,64,128,256\}$;
plot test accuracy against **inference** FLOPs and draw the Pareto frontier, with plain
MLPs of several widths on the same axes. Fix epochs. Three seeds everywhere.

**Scope.** *Two days*: the $(r, \text{hidden})$ grid and the frontier. *Five days*: add the
train-then-factorise arm, and plot the singular-value spectrum of the trained
$\mathbf{W}$ — if it decays fast, low rank should be nearly free, and you can check whether
that prediction holds.

<a id="q2"></a>
## 2. Ternary weights — what does 1.58 bits cost on tabular data?

**The idea.** BitNet b1.58 constrains every weight to $\{-1, 0, +1\}$ with one scale per
matrix, so a matrix product becomes additions and subtractions instead of multiplications.
The forward quantises; the backward pretends the quantiser was the identity (a
straight-through estimator), which is exactly the kind of custom op you hand-wrote in
[Exercise 01](../exercises/q01_activations.ipynb).

**Readings**
- Ma et al., *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits*, 2024 —
  [arXiv:2402.17764](https://arxiv.org/abs/2402.17764).
- Courbariaux, Bengio & David, *BinaryConnect*, NeurIPS 2015 —
  [arXiv:1511.00363](https://arxiv.org/abs/1511.00363).
- Bengio, Léonard & Courville, *Estimating or Propagating Gradients Through Stochastic
  Neurons*, 2013 — [arXiv:1308.3432](https://arxiv.org/abs/1308.3432). Where the
  straight-through estimator comes from, and why it is not obviously legitimate.

**The open question.** *How much accuracy does ternarisation cost on 108 standardised and
one-hot features — and can widening a ternary network recover it at a lower effective
cost than the dense baseline?*

**What to implement.** A `ternarize` op with a straight-through (or clipped
straight-through) `_backward`, following the pattern in
[`bert_cpu/engine.py`](../bert_cpu/engine.py); a `TernaryLinear` that quantises its weight
inside `forward` while keeping a full-precision shadow copy for the update; and an
**effective FLOP model** — the engine's counter charges a ternary matmul at full MAC
price, so you must state and justify your own accounting (e.g. one add per element instead
of a multiply-add).

**Protocol.** Baseline versus ternary at `hidden` $\in \{64, 128, 256, 512\}$. Report both
the counter's number and your effective number, clearly labelled. Track the fraction of
zeros in the quantised weights and the distribution of the scale. Ablate the scale
granularity (per matrix versus per output unit).

**Scope.** *Two days*: the STE op, gradient-check it against the identity surrogate,
ternarise the first layer. *Five days*: both layers, the width sweep, and an honest
discussion of what "FLOP" even means once multiplies disappear.

<a id="q3"></a>
## 3. Sparse training — is it better to prune, or to prune and grow?

**The idea.** Most of the $109 \times 64$ matrix may be unnecessary. You can find that out
three ways: train dense then prune the small weights; start from a random sparse mask and
never change it; or start sparse and **evolve** the mask during training, dropping the
smallest weights and growing new ones (SET, RigL). The three cost very different amounts to
train and end at the same sparsity.

**Readings**
- Mocanu et al., *Scalable training of artificial neural networks with adaptive sparse
  connectivity inspired by network science*, Nature Communications 2018 —
  [arXiv:1707.04780](https://arxiv.org/abs/1707.04780). SET; a NumPy-sized algorithm.
- Evci et al., *Rigging the Lottery: Making All Tickets Winners*, ICML 2020 —
  [arXiv:1911.11134](https://arxiv.org/abs/1911.11134). Grows by gradient magnitude.
- Frankle & Carbin, *The Lottery Ticket Hypothesis*, ICLR 2019 —
  [arXiv:1803.03635](https://arxiv.org/abs/1803.03635).
- Sanh, Wolf & Rush, *Movement Pruning*, NeurIPS 2020 —
  [arXiv:2005.07683](https://arxiv.org/abs/2005.07683). Prune by *motion*, not magnitude.

**The open question.** *At 80–95 % sparsity, does dynamic sparse training beat pruning a
dense-trained network at equal training FLOPs?* And the small-scale version of the lottery
ticket question: *does a sparse mask found by pruning still work when rewound to the
initial weights?*

**What to implement.** A mask per weight matrix, applied in the forward pass
(`w * mask_tensor`, with the mask a constant); magnitude pruning; a SET-style
prune-and-grow step every $k$ epochs (`on_epoch` in `run_adult` is the hook for it). Decide
and justify how a masked matmul is counted — the engine charges it densely.

**Protocol.** Sparsity $\in \{0.5, 0.8, 0.9, 0.95, 0.98\}$ across three regimes:
dense→prune→fine-tune, static random sparse, dynamic sparse. Equal *training* FLOP budget
across regimes (the dense phase counts). Report accuracy against effective inference cost.

**Scope.** *Two days*: magnitude pruning and static sparsity. *Five days*: add SET/RigL
growth and the rewind experiment, and report how much the answer moves between seeds — at
this scale, masks are noisy.

<a id="q4"></a>
## 4. Conditional compute — does routing pay off on tabular data?

**The idea.** Instead of one hidden layer that every row passes through, hold $E$ smaller
experts and let a tiny router send each row to **one** of them. Parameters go up, the
compute *per row* stays flat. This is the idea behind sparse mixture-of-experts models; on
a 108-feature census table, it is entirely unclear whether there is anything for a router
to specialise on.

**Readings**
- Shazeer et al., *Outrageously Large Neural Networks: The Sparsely-Gated
  Mixture-of-Experts Layer*, ICLR 2017 —
  [arXiv:1701.06538](https://arxiv.org/abs/1701.06538). Read the load-balancing loss
  carefully; you will need it.
- Fedus, Zoph & Shazeer, *Switch Transformers*, JMLR 2022 —
  [arXiv:2101.03961](https://arxiv.org/abs/2101.03961). Top-1 routing, and why it is enough.
- Raposo et al., *Mixture-of-Depths*, 2024 —
  [arXiv:2404.02258](https://arxiv.org/abs/2404.02258). Routing as a compute budget rather
  than as capacity.

**The open question.** *Does a top-1 mixture of experts beat a dense MLP with the same
active FLOPs — and if it does, is the router learning structure in the data, or is it a
lucky ensemble?*

**What to implement.** $E$ expert layers, a linear router over the input, hard top-1
dispatch (gather the rows per expert with the engine's indexing, run each expert on its
slice, scatter the results back), the auxiliary load-balancing loss inside `loss_fn`, and a
usage report: rows per expert, expert usage entropy, and expert usage broken down by label.

**Protocol.** $E \in \{2,4,8\}$ against dense baselines matched on *active* FLOPs; with and
without the balancing loss (routers collapse without it — show that). Fix epochs.

**Scope.** *Two days*: routing, balancing, usage statistics. *Five days*: add soft top-$k$
routing as a control, a capacity limit with dropped rows, and an interpretability pass —
does an expert own the high-education rows, or the married ones?

<a id="q5"></a>
## 5. Star versus sum — is a second-order branch worth its FLOPs?

**The idea.** [Exercise 02](../exercises/q02_rewrite_the_stars.ipynb) showed that
$\mathrm{act}(\mathbf{W}_1^{\top}\tilde{\mathbf{x}}) \odot
(\mathbf{W}_2^{\top}\tilde{\mathbf{x}})$ carries $x_i x_j$ interaction terms a single
linear layer cannot express — and you verified the mixed second derivative is non-zero.
But it costs two branches. On a table full of one-hot indicators, where interactions
between categories are exactly what a gradient-boosted tree would exploit, is that a good
trade?

**Readings**
- Ma, Dai, Bai, Wang & Fu, *Rewrite the Stars*, CVPR 2024 —
  [arXiv:2403.19967](https://arxiv.org/abs/2403.19967).
- Shazeer, *GLU Variants Improve Transformer*, 2020 —
  [arXiv:2002.05202](https://arxiv.org/abs/2002.05202). The same shape, gated.
- Dauphin, Fan, Auli & Grangier, *Language Modeling with Gated Convolutional Networks*,
  ICML 2017 — [arXiv:1612.08083](https://arxiv.org/abs/1612.08083).
- Grinsztajn, Oyallon & Varoquaux, *Why do tree-based models still outperform deep learning
  on tabular data?*, NeurIPS 2022 —
  [arXiv:2207.08815](https://arxiv.org/abs/2207.08815). Read this for what tabular data
  rewards.

**The open question.** *At equal FLOPs — a star layer of width $h$ against a plain layer of
width $2h$ — which wins on Adult, and does the answer depend on the width?*

**What to implement.** Lift `StarLinear` (and, if you go further, the GLU family from
[q03](../exercises/q03_gated_linear_units.ipynb)) out of the exercise notebooks into the
MLP, and build the FLOP-matched comparison carefully — a star layer at width $h$ costs
about the same as a plain layer at width $2h$, so those are the pairs to compare.

**Protocol.** $h \in \{16, 32, 64, 128\}$ for the star, paired with plain layers at $2h$.
Same epochs, three seeds, accuracy against training and inference FLOPs.

**Scope.** *Two days*: the FLOP-matched pairs at three widths. *Five days*: add the GLU
variants, and test *where* the advantage (if any) lives — retrain with only the continuous
features, then with only the one-hot block, and see whether the star's edge tracks the
categorical interactions as the theory suggests.

<a id="q6"></a>
## 6. Activation sparsity — how much compute could you skip?

**The idea.** After a ReLU, many hidden units are exactly zero; anything multiplied by them
in the next layer is wasted work. At LLM scale this "lazy neuron" phenomenon is dramatic
(>90 % of units inactive per token) and is the basis of a family of inference speedups. At
width 64 on a census table, nobody knows.

**Readings**
- Zhang et al., *ReLU² Wins: Discovering Efficient Activation Functions for Sparse LLMs*,
  2024 — [arXiv:2402.03804](https://arxiv.org/abs/2402.03804).
- Li et al., *The Lazy Neuron Phenomenon: On Emergence of Activation Sparsity in
  Transformers*, ICLR 2023 — [arXiv:2210.06313](https://arxiv.org/abs/2210.06313).
- Mirzadeh et al., *ReLU Strikes Back*, ICLR 2024 —
  [arXiv:2310.04564](https://arxiv.org/abs/2310.04564).
- So et al., *Primer: Searching for Efficient Transformers for Language Modeling*, NeurIPS 2021 —
  [arXiv:2109.08668](https://arxiv.org/abs/2109.08668). Where squared ReLU comes from.

**The open question.** *Which activation gives the most skippable compute at equal
accuracy — and does activation sparsity grow with width here, as it does at scale?*

**What to implement.** ReLU² (as a composed op, or as a fused op with a hand-written
backward — compare the FLOP counts of the two, that is a finding in itself); a sparsity
meter (fraction of hidden units below a threshold $\tau$, per row); and an *effective*
inference FLOP count that charges only the active units, with $\tau$ stated.

**Protocol.** $\{\mathrm{ReLU}, \mathrm{ReLU}^2, \mathrm{GELU}, \mathrm{SiLU}\}$ across
`hidden` $\in \{64, 256, 1024\}$. Report accuracy, sparsity at two thresholds, and
effective cost. Note that GELU and SiLU are never exactly zero — that is precisely why
the threshold has to be part of the claim.

**Scope.** *Two days*: the meter and four activations at one width. *Five days*: the width
sweep, plus a **top-$k$** experiment — at inference, keep only the $k$ largest activations
and zero the rest; find where accuracy breaks, which tells you how much of the layer was
doing real work.

<a id="q7"></a>
## 7. Learned activation mixtures — do the coefficients earn their keep?

**The idea.** [Exercise 04](../exercises/q04_learnable_activations.ipynb) built
$\varphi(z) = \alpha_1 \mathrm{ReLU}(z) + \alpha_2 \mathrm{GELU}(z) + \alpha_3
\mathrm{SiLU}(z)$ with trainable $\alpha_k$ — and then never trained it. Here you do. Three
extra scalars is a negligible FLOP cost, so this is the cleanest possible test of a
question that keeps coming back in the literature: is a learned activation better than the
best fixed one, or does it just add hyper-parameters?

**Readings**
- Wang, Wang, Xia, Shen & Zhong, *More Expressive Feedforward Layers: Part I.
  Token-Adaptive Mixing of Activations*, 2026 — the paper Exercise 04 is built on.
- He, Zhang, Ren & Sun, *Delving Deep into Rectifiers* (PReLU), ICCV 2015 —
  [arXiv:1502.01852](https://arxiv.org/abs/1502.01852). The first widely used learned
  activation.
- Ramachandran, Zoph & Le, *Searching for Activation Functions*, 2017 —
  [arXiv:1710.05941](https://arxiv.org/abs/1710.05941). How Swish was found, and how small
  the gains were.
- Liu et al., *KAN: Kolmogorov–Arnold Networks*, 2024 —
  [arXiv:2404.19756](https://arxiv.org/abs/2404.19756). The maximalist version: make every
  activation learnable, and read the debate that followed.

**The open question.** *Is the mixture better than the best single activation by more than
the seed noise — and if the coefficients converge somewhere interesting, do they converge
to the same place from different seeds?*

**What to implement.** Drop `LearnableActivation` and `NormalizedLearnableActivation` from
q04 into the MLP as the hidden nonlinearity and train them. Log the coefficient trajectory
per epoch.

**Protocol.** This effect is small, so use **five seeds or more**. Compare: fixed ReLU,
fixed GELU, fixed SiLU, unconstrained mixture, softmax-normalised mixture. Same epochs,
same width. Then answer the question the report must answer: *would those parameters have
been better spent on two more hidden units?* (Run that arm.)

**Scope.** *Two days*: the comparison and the coefficient trajectories. *Five days*: add a
per-layer and a per-unit mixture (the parameter count grows — track it), and check whether
the learned mixture transfers: freeze the coefficients found on Adult and use them on the
q06 task.

<a id="q8"></a>
## 8. Loss shape under class imbalance

**The idea.** 76 % of Adult rows are `<=50K`. Cross entropy treats every row alike, and the
model can score 0.76 by ignoring the minority class entirely. A family of losses exists to
push back — focal down-weights easy examples, label smoothing softens the targets, PolyLoss
re-weights the leading term of the cross-entropy expansion — and all of them cost
essentially nothing in FLOPs, which makes this the cheapest accuracy on offer *if* it
works.

**Readings**
- Lin et al., *Focal Loss for Dense Object Detection*, ICCV 2017 —
  [arXiv:1708.02002](https://arxiv.org/abs/1708.02002).
- Szegedy et al., *Rethinking the Inception Architecture*, CVPR 2016 —
  [arXiv:1512.00567](https://arxiv.org/abs/1512.00567). §7: label smoothing.
- Müller, Kornblith & Hinton, *When Does Label Smoothing Help?*, NeurIPS 2019 —
  [arXiv:1906.02629](https://arxiv.org/abs/1906.02629). Read this for the representation
  geometry argument — it tells you what to measure.
- Leng et al., *PolyLoss*, ICLR 2022 —
  [arXiv:2204.12511](https://arxiv.org/abs/2204.12511).

**The open question.** *Which loss buys the most on the minority class at equal FLOPs — and
does it change the geometry of the hidden representation the way Müller et al. claim?*

**What to implement.** Focal, label-smoothed and Poly-1 cross entropy on top of the
engine's `softmax` (all three are a few lines in `loss_fn`); class-weighted cross entropy as
a reference point. Metrics beyond accuracy: minority-class precision and recall, balanced
accuracy, and a reliability diagram for calibration.

**Protocol.** Sweep the one hyper-parameter each loss has ($\gamma$, $\varepsilon$,
$\epsilon_1$). Report the full metric table — accuracy alone will hide the effect. Then
take the trained hidden layer, label the rows by class, and compute
`silhouette_score` from `benchmark.py` on a sample: does label smoothing really tighten the
clusters?

**Scope.** *Two days*: three losses at default settings, full metric table. *Five days*:
the sweeps, the silhouette study, and calibration — smoothing is known to trade accuracy
for calibration, so check whether that shows up here.

<a id="q9"></a>
## 9. R-Drop versus simply training longer

**The idea.** Dropout makes the network stochastic; R-Drop runs the *same* input through it
**twice** and adds a symmetric KL term pushing the two output distributions together. It
reliably helps in the papers — but it doubles the forward cost of every step. Nobody in
those papers spends the same compute on plain dropout and more epochs, which is the
comparison that decides whether the idea is worth anything on a budget.

**Readings**
- Liang et al., *R-Drop: Regularized Dropout for Neural Networks*, NeurIPS 2021 —
  [arXiv:2106.14448](https://arxiv.org/abs/2106.14448).
- Srivastava, Hinton, Krizhevsky, Sutskever & Salakhutdinov, *Dropout*, JMLR 15(1), 2014.
- Gal & Ghahramani, *Dropout as a Bayesian Approximation*, ICML 2016 —
  [arXiv:1506.02142](https://arxiv.org/abs/1506.02142). Why two stochastic forwards are two
  samples from a posterior.

**The open question.** *At equal total training FLOPs, does R-Drop beat plain dropout
trained for twice as many steps?* A run of this comparison with three seeds already shows
R-Drop ahead by +0.002 accuracy for 1.9× the training FLOPs at equal **epochs** — your job
is the equal-**FLOPs** version of that table, which may well reverse it.

**What to implement.** `nn.Dropout` is a stub at
[`bert_cpu/nn.py`](../bert_cpu/nn.py) (line ~281) — implementing inverted dropout with a
working `train()`/`eval()` switch is part of the task, and `Module.train` already exists to
drive it. Then the R-Drop objective in `loss_fn`: two forwards, mean cross entropy, plus
$\alpha \cdot \tfrac{1}{2}\left[\mathrm{KL}(p_1\|p_2) + \mathrm{KL}(p_2\|p_1)\right]$.

**Protocol.** Three arms at a **fixed training-FLOP budget**: plain, dropout, R-Drop (which
gets roughly half the epochs). Sweep $p \in \{0.1, 0.2, 0.5\}$ and $\alpha \in \{0.5, 1, 5\}$.
Report the equal-epoch table too, so the difference between the two accountings is visible
— that contrast *is* the result.

**Scope.** *Two days*: Dropout (gradient-check it in eval mode), then the three arms.
*Five days*: the sweeps, plus the question of whether R-Drop's gain is really about
consistency or just about seeing each row twice — build the control that answers it.

<a id="q10"></a>
## 10. Mixup where interpolation is meaningless

**The idea.** Mixup trains on convex combinations of pairs of examples and their labels,
$\tilde{x} = \lambda x_i + (1-\lambda)x_j$. On images that yields a ghostly blend; on this
dataset, 95 of the 108 features are one-hot indicators, so mixing produces a row that is
0.7 "Bachelors" and 0.3 "Doctorate" — a person who does not exist. It still might
regularise. Manifold mixup sidesteps the objection by mixing the *hidden* layer instead.

**Readings**
- Zhang, Cissé, Dauphin & Lopez-Paz, *mixup: Beyond Empirical Risk Minimization*, ICLR
  2018 — [arXiv:1710.09412](https://arxiv.org/abs/1710.09412).
- Verma et al., *Manifold Mixup*, ICML 2019 —
  [arXiv:1806.05236](https://arxiv.org/abs/1806.05236).
- Grinsztajn, Oyallon & Varoquaux, *Why do tree-based models still outperform deep learning
  on tabular data?*, NeurIPS 2022 —
  [arXiv:2207.08815](https://arxiv.org/abs/2207.08815). §4 on uninformative features and
  rotation invariance is the theoretical backdrop for this question.

**The open question.** *Does mixup regularise a network on one-hot tabular data — and is
hidden-space mixing better precisely because it does not fabricate impossible rows?*

**What to implement.** Input mixup inside `loss_fn` (draw $\lambda \sim
\mathrm{Beta}(\alpha, \alpha)$, permute the batch, mix inputs and one-hot labels — you will
need the soft-label form of cross entropy, which is three lines on top of `softmax`);
manifold mixup (mix after the first layer); and the ablation that makes this a research
question rather than a replication: **mix only the continuous features**, leaving the
one-hot block untouched.

**Protocol.** $\alpha \in \{0.1, 0.2, 0.4, 1.0\}$ × {input, hidden, continuous-only}. Fixed
epochs (mixup costs almost nothing per step). Report accuracy and calibration.

**Scope.** *Two days*: input mixup and the $\alpha$ sweep. *Five days*: all three variants,
plus a look at what mixup does to the minority class — averaging labels in an imbalanced
problem is not neutral.

<a id="q11"></a>
## 11. Optimisers at an equal training budget

**The idea.** The baseline uses Adam because Adam is what everyone uses. Newer optimisers
claim to reach the same loss in fewer steps — Lion with a sign-based update and less state,
Sophia with a cheap curvature estimate, Muon by orthogonalising the momentum matrix with a
few Newton–Schulz iterations. All of them were argued at language-model scale. Your weight
matrices are $109 \times 64$ and $65 \times 2$.

**Readings**
- Loshchilov & Hutter, *Decoupled Weight Decay Regularization* (AdamW), ICLR 2019 —
  [arXiv:1711.05101](https://arxiv.org/abs/1711.05101).
- Chen et al., *Symbolic Discovery of Optimization Algorithms* (Lion), NeurIPS 2023 —
  [arXiv:2302.06675](https://arxiv.org/abs/2302.06675).
- Liu et al., *Sophia*, ICLR 2024 — [arXiv:2305.14342](https://arxiv.org/abs/2305.14342).
- Jordan, *Muon: An optimizer for hidden layers in neural networks*, 2024
  ([kellerjordan.github.io/posts/muon](https://kellerjordan.github.io/posts/muon/)) and Liu
  et al., *Muon is Scalable for LLM Training*, 2025 —
  [arXiv:2502.16982](https://arxiv.org/abs/2502.16982).
- Defazio et al., *The Road Less Scheduled*, NeurIPS 2024 —
  [arXiv:2405.15682](https://arxiv.org/abs/2405.15682). If you would rather attack the
  schedule than the optimiser.

**The open question.** *Which optimiser reaches the baseline's 0.8543 for the fewest
training FLOPs at this size — and does Muon's orthogonalised update do anything useful on a
matrix this small?*

**What to implement.** Subclass `optim.Optimizer` (read `SGD` and `Adam` in
[`bert_cpu/optim.py`](../bert_cpu/optim.py) first). Lion is a five-line update; Muon's
Newton–Schulz orthogonalisation is about ten lines of NumPy. Count the optimiser's own
arithmetic — Muon's iteration is five extra matmuls per step on the weight matrix, which is
small here but not zero, and saying so is part of a fair comparison.

**Protocol.** **Tune the learning rate for every optimiser** over a log grid — an untuned
comparison is worthless and will be marked as such. Then plot test accuracy against
*cumulative* training FLOPs and report FLOPs-to-target (the budget needed to first reach
0.8543). Three seeds at the best learning rate.

**Scope.** *Two days*: SGD, momentum, Adam, with the grids. *Five days*: add Lion and Muon,
the cost accounting, and a look at whether the winner changes when the model is wider —
the papers' claims are all about scale, so test the trend, not just the point.

<a id="q12"></a>
## 12. Distil a wide teacher into a tiny student

**The idea.** Train a wide network, then train a small one to match its *soft* output
distribution rather than the hard labels. The classic claim is that the small network ends
up better than if it had been trained on the labels directly — the teacher's probabilities
carry "dark knowledge" about which rows are ambiguous. The inference cost is the student's
alone, which is the whole point.

**Readings**
- Hinton, Vinyals & Dean, *Distilling the Knowledge in a Neural Network*, 2015 —
  [arXiv:1503.02531](https://arxiv.org/abs/1503.02531).
- Furlanello et al., *Born-Again Neural Networks*, ICML 2018 —
  [arXiv:1805.04770](https://arxiv.org/abs/1805.04770). Distilling into the *same*
  architecture, which should not help, and does.
- Beyer et al., *Knowledge distillation: A good teacher is patient and consistent*, CVPR
  2022 — [arXiv:2106.05237](https://arxiv.org/abs/2106.05237). The paper that explains why
  most distillation experiments are run for too few epochs.
- Müller, Kornblith & Hinton, *When Does Label Smoothing Help?*, NeurIPS 2019 —
  [arXiv:1906.02629](https://arxiv.org/abs/1906.02629). §4: smoothing the teacher can
  *destroy* the information distillation relies on. The control your experiment needs.

**The open question.** *At a fixed inference budget, is a distilled small MLP better than
the same MLP trained directly — and does the conclusion survive counting the teacher's
training FLOPs in the total?*

**What to implement.** A teacher (`hidden=512`), then students at `hidden` $\in \{4, 8, 16,
32\}$ trained on $\lambda\,\mathrm{CE}(\text{hard}) + (1-\lambda)\,T^2\,
\mathrm{KL}(\text{teacher}_T \,\|\, \text{student}_T)$ — the teacher's logits are constants
(`requires_grad=False`), computed once before training, so the student's step stays cheap.

**Protocol.** Two accountings, both reported: inference-only (what you would deploy) and
total training FLOPs (teacher + student). Sweep $T \in \{1,2,4\}$ and $\lambda$. The control
arm that makes this a real experiment: the same student trained with **label smoothing**
matched to the teacher's average confidence — if that matches distillation, the dark
knowledge was not the point.

**Scope.** *Two days*: teacher, one student width, the $T$ sweep. *Five days*: the width
sweep, self-distillation (teacher and student the same size), and the smoothing control.

<a id="q13"></a>
## 13. How much of the training set can you throw away?

**The idea.** Training FLOPs scale linearly with rows. If half of Adult's 26 049 training
rows are redundant, half the training compute is too — provided you can tell *which* half
without spending more than you save. Scores like EL2N (the norm of the error early in
training) and forgetting counts (how often a row flips from correct to wrong) claim to do
exactly that, and the literature reports a twist: which examples you should keep depends on
how many you can afford.

**Readings**
- Sorscher et al., *Beyond neural scaling laws: beating power law scaling via data
  pruning*, NeurIPS 2022 —
  [arXiv:2206.14486](https://arxiv.org/abs/2206.14486). The keep-easy/keep-hard reversal is
  the hypothesis to test.
- Toneva et al., *An Empirical Study of Example Forgetting*, ICLR 2019 —
  [arXiv:1812.05159](https://arxiv.org/abs/1812.05159).
- Paul, Ganguli & Dziugaite, *Deep Learning on a Data Diet*, NeurIPS 2021 —
  [arXiv:2107.07075](https://arxiv.org/abs/2107.07075). GraNd and EL2N.

**The open question.** *How much of the training set can go, and by which score, before test
accuracy falls below the baseline — and does the best pruning rule flip between keeping easy
and keeping hard rows as the budget shrinks, as Sorscher et al. predict?*

**What to implement.** EL2N (train a few epochs, record $\|p_i - y_i\|_2$ per row) and
forgetting counts (track per-row correctness across epochs — `on_epoch` is the hook).
`run_adult(..., subset=indices)` already accepts the kept rows. **Count the scoring cost**:
EL2N needs a short training run, and a data diet that costs more than it saves is not a
saving.

**Protocol.** Keep fractions $\{0.1, 0.2, 0.4, 0.6, 0.8, 1.0\}$ × {random, EL2N-hard,
EL2N-easy, forgetting}. Accuracy against total training FLOPs (scoring included). Check the
stability of the ranking across seeds — if two seeds disagree about which rows are hard, the
score is measuring noise.

**Scope.** *Two days*: random versus EL2N at five fractions. *Five days*: add forgetting
scores, the seed-stability analysis, and the easy/hard reversal at small budgets.

<a id="q14"></a>
## 14. Can you learn a representation without the labels?

**The idea.** Adult has 32 561 training rows and every one comes with a label — but suppose
it did not. SCARF learns a representation by corrupting a row (replace a random subset of
features with values drawn from that feature's own empirical distribution) and pulling the
corrupted view towards the original and away from the rest of the batch. Then a *linear*
probe on the frozen representation does the classification. The question is whether that
detour is ever worth its compute.

**Readings**
- Bahri, Jiang, Tay & Metzler, *SCARF: Self-Supervised Contrastive Learning using Random
  Feature Corruption*, ICLR 2022 —
  [arXiv:2106.15147](https://arxiv.org/abs/2106.15147).
- Zbontar, Jing, Misra, LeCun & Deny, *Barlow Twins*, ICML 2021 —
  [arXiv:2103.03230](https://arxiv.org/abs/2103.03230). A loss with no negatives at all —
  a much cheaper objective, which matters here.
- Bardes, Ponce & LeCun, *VICReg*, ICLR 2022 —
  [arXiv:2105.04906](https://arxiv.org/abs/2105.04906).

**The open question.** *Does corruption-contrastive pretraining beat supervised training
per **total** FLOP — and if it does not at 100 % of the labels, does it at 1 %?* (That
second half is where self-supervision is supposed to earn its keep, and it is one line of
code away once the first half runs.)

**What to implement.** The corruption (sample replacement values from the column's own
empirical marginal), an encoder, a small projection head, InfoNCE over the batch
(temperature $\tau$), then freeze and fit a single `nn.Linear` probe with `cross_entropy`.
Mini-batching matters here — the contrastive loss needs negatives in the batch, so this is
the one Adult question that is not full-batch.

**Protocol.** Pretrain for $E$ epochs, probe, and compare against the supervised baseline
**at the same total FLOPs** (pretraining + probe). Sweep the corruption rate. Then the
label-scarce arm: probe on 1 %, 10 %, 100 % of the labels, supervised baseline likewise.

**Scope.** *Two days*: corruption + InfoNCE + probe at one setting, with the honest total.
*Five days*: the corruption sweep, Barlow Twins' loss as a cheaper alternative (no
negatives — does the FLOP accounting change the winner?), and the label-scarce regime.

<a id="q15"></a>
## 15. The shape of an embedding table

**The idea.** [Exercise 06](../exercises/q06_learn_embedding.ipynb) fixed $D = 32$, $K = 5$
negatives and 100 epochs because those numbers work, not because they are optimal. Every
one of them is a FLOP knob, and the metric — the silhouette of three word groups — is
noisy enough that measuring properly is most of the work. There is also a theory to test:
Levy & Goldberg showed skip-gram with negative sampling is implicitly factorising a shifted
PMI matrix, which makes the right dimension a question about that matrix's spectrum rather
than a hyper-parameter to tune blindly.

**Readings**
- Mikolov, Sutskever, Chen, Corrado & Dean, *Distributed Representations of Words and
  Phrases*, NeurIPS 2013 — [arXiv:1310.4546](https://arxiv.org/abs/1310.4546). §2.2 (the
  noise distribution) and §2.3 (subsampling frequent words) are the two knobs to add.
- Levy & Goldberg, *Neural Word Embedding as Implicit Matrix Factorization*, NeurIPS 2014.
- Yin & Shen, *On the Dimensionality of Word Embedding*, NeurIPS 2018 —
  [arXiv:1812.04224](https://arxiv.org/abs/1812.04224). A bias-variance account of how to
  pick $D$; it makes a prediction you can check.
- Press & Wolf, *Using the Output Embedding to Improve Language Models*, EACL 2017 —
  [arXiv:1608.05859](https://arxiv.org/abs/1608.05859). Tying the two tables.

**The open question.** *What is the FLOP-optimal $(D, K, \text{epochs})$ for a target
silhouette on this corpus — and does tying the input and output tables halve the parameters
for free?*

**What to implement.** Mostly sweeps: `run_sgns(dim=..., neg_k=..., epochs=...,
tie_tables=True)` already supports all of it. Add frequent-word subsampling (drop a token
with probability $1 - \sqrt{t/f(w)}$) and, if you go for the five-day version, build the
PMI matrix of this corpus explicitly and compare its truncated SVD with the learned vectors.

**Protocol.** $D \in \{8,16,32,64,128\}$ × $K \in \{2,5,10,20\}$ at fixed epochs, then the
epochs axis. **Five seeds minimum** — the silhouette is computed over 18 words and moves
between seeds. Report the per-word breakdown, not only the mean, and sanity-check with
nearest neighbours: a table that scores well but returns nonsense neighbours is telling you
the metric is too small.

**Scope.** *Two days*: the $D \times K$ grid. *Five days*: add tying, subsampling, the
epochs axis (q06's own note that *more* epochs can lower the silhouette is worth confirming
or refuting), and the PMI-factorisation comparison.

---

## What you hand in

A report of **six to ten pages**, plus the code that produced every number in it. Structure
it like this — the sections are not optional, and they are in this order because that is
the order in which a reader decides whether to believe you.

1. **The question.** One paragraph. Which of the fifteen, and what you took it to mean.
2. **What the literature says.** The papers you read, what they claim, and — crucially —
   what they predict for *this* setting: a 108-feature tabular problem or an 886-word
   corpus, in float64, on a CPU. Most of these results were obtained three orders of
   magnitude away from here.
3. **Hypothesis.** A falsifiable sentence, written **before** you ran the experiments. For
   example: "ternarising the first layer costs less than 0.5 points of test accuracy at
   `hidden=64`, and the loss vanishes by `hidden=256`."
4. **What you implemented.** Which files, roughly how many lines, and how a reader runs it.
   If you implemented an engine op, show the gradient check.
5. **Protocol.** What you varied, what you held fixed, how many seeds, what the budget was
   (epochs or FLOPs — say which), and what you tuned versus what you took from the baseline.
6. **Results.** The `summarize()` table with the baseline in it, and the `pareto()` figure.
   Every claim in prose must point at a row or a point.
7. **Threats to validity.** The FLOP counter charges elementwise ops approximately and
   masked or quantised matmuls at full price; one dataset is not evidence about datasets;
   a hyper-parameter you did not tune may be carrying the result. Name yours.
8. **What remains open.** The next experiment you would run, and what it would decide.
9. **Reproduction.** Exact commands, seeds, and the wall-clock cost of the whole study.

## How it is graded

| weight | what is assessed |
|---|---|
| **35 %** | **Measurement rigour** — seeds, error bars, the right thing held fixed, FLOPs reported beside every score, no claim inside the noise |
| **20 %** | **Implementation correctness** — does the code do what the paper says? Gradient checks for new ops, sanity checks for new losses, a baseline you reproduced before changing anything |
| **15 %** | **Engagement with the literature** — not a summary of the papers, but what they predicted here and whether it happened |
| **15 %** | **Honesty and clarity** — a clean negative result scores full marks; an overstated positive one does not |
| **15 %** | **Reproducibility** — someone else can run your study from your repository and get your table |

Three things fail the project regardless of the rest: **a single seed**, **an accuracy
without its FLOP cost**, and **a claim whose difference is smaller than its error bar**.

## A closing note

The honest prior is that for most of these fifteen questions the answer at this scale is
*"no, and here is the measurement that shows it."* That is not a disappointing outcome — it
is the normal one, and reporting it clearly is exactly the skill this project exists to
teach. The ideas in these papers were found at a scale where a 0.2 % gain is worth
millions; discovering that the same idea is invisible on 26 049 rows of census data tells
you something true about the idea, about the scale, and about how much of the literature
survives contact with a small problem.

Work on one question. Measure it properly. Say what you found.